# Baseline 1a: Tìm kiếm Từ khóa bằng Mô hình Không gian Vector TF-IDF & Cosine Similarity

## 1. Lý thuyết toán học nâng cao (Dành cho Vấn đáp)

Mô hình không gian vector biểu diễn văn bản dưới dạng các vector số trong không gian nhiều chiều, nơi mỗi chiều tương ứng với một từ (term) trong từ điển.

### 1.1 Tần suất từ - TF (Term Frequency)
Đo mức độ quan trọng của từ $t$ trong văn bản $d$. Nếu từ xuất hiện càng nhiều, trọng số càng lớn.
$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$
*(Trong đó $f_{t,d}$ là số lần xuất hiện của từ $t$ trong văn bản $d$, mẫu số là tổng số từ của văn bản $d$).*

### 1.2 Nghịch đảo tần suất văn bản - IDF (Inverse Document Frequency)
Đo mức độ phổ biến của từ $t$ trên toàn bộ tập dữ liệu (corpus) $D$. Từ xuất hiện ở quá nhiều văn bản (ví dụ: 'the', 'is') sẽ ít mang thông tin phân biệt $\to$ IDF thấp. Từ hiếm gặp $\to$ IDF cao.
$$\text{IDF}(t, D) = \log \frac{|D|}{1 + |\{d \in D : t \in d\}|}$$
*(Mẫu số cộng thêm 1 để tránh lỗi chia cho 0 khi từ không xuất hiện ở bất kỳ văn bản nào - Smooth IDF).*

### 1.3 Trọng số TF-IDF
Kết hợp độ quan trọng cục bộ (TF) và độ phổ biến toàn cục (IDF):
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

### 1.4 Độ tương đồng Cosine (Cosine Similarity)
Đo góc giữa hai vector câu hỏi (query $q$) và tài liệu (document $d$). Cosine bỏ qua độ dài văn bản, chỉ quan tâm đến hướng vector (tương quan chủ đề).
$$\cos(\theta) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \|\vec{d}\|} = \frac{\sum_{i} q_i d_i}{\sqrt{\sum_i q_i^2} \sqrt{\sum_i d_i^2}}$$
*(Tử số là tích vô hướng, mẫu số là tích độ dài L2-Norm của hai vector).*

## 2. Khởi tạo Toy Corpus (Dữ liệu mẫu)
Dựng 5 câu văn mẫu chứa các thuật ngữ tài chính về doanh thu (revenue, sales) và chi tiêu (capex, purchases) để làm rõ điểm yếu **Lexical Gap**.

In [11]:
import numpy as np
import pandas as pd
import re

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

for k, v in corpus.items():
    print(f"{k}: {v}")

doc1: Apple Net sales in fiscal year 2023 were $383,285 million.
doc2: Apple payments for acquisition of property plant and equipment were $10,959 million.
doc3: Amazon net sales were $574,785 million in fiscal year 2023.
doc4: Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.
doc5: NVIDIA net income was $29,760 million in fiscal year 2024.


## 3. Lập chỉ mục TF-IDF thủ công (Step-by-Step Matrix Construction)
Chúng ta sẽ tự viết code xây dựng từ điển, tính TF, IDF và ma trận TF-IDF thô để in ra chi tiết các bước tính toán trung gian.

In [12]:
# 3.1 Tokenization & Lowercasing
def tokenize(text):
    text = text.lower()
    # Xóa ký tự đặc biệt, chỉ giữ chữ và số và dấu đô la
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
print("[Log] Tokenized Documents:")
for k, v in tokenized_docs.items():
    print(f"  - {k}: {v}")

# 3.2 Xây dựng Từ điển Từ vựng (Vocabulary)
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))
print(f"\n[Log] Vocabulary Size: {len(vocabulary)} terms")
print(f"Vocabulary: {vocabulary}")

[Log] Tokenized Documents:
  - doc1: ['apple', 'net', 'sales', 'in', 'fiscal', 'year', '2023', 'were', '$383285', 'million']
  - doc2: ['apple', 'payments', 'for', 'acquisition', 'of', 'property', 'plant', 'and', 'equipment', 'were', '$10959', 'million']
  - doc3: ['amazon', 'net', 'sales', 'were', '$574785', 'million', 'in', 'fiscal', 'year', '2023']
  - doc4: ['amazon', 'purchases', 'of', 'property', 'and', 'equipment', 'in', 'fiscal', 'year', '2023', 'were', '$52729', 'million']
  - doc5: ['nvidia', 'net', 'income', 'was', '$29760', 'million', 'in', 'fiscal', 'year', '2024']

[Log] Vocabulary Size: 28 terms
Vocabulary: ['$10959', '$29760', '$383285', '$52729', '$574785', '2023', '2024', 'acquisition', 'amazon', 'and', 'apple', 'equipment', 'fiscal', 'for', 'in', 'income', 'million', 'net', 'nvidia', 'of', 'payments', 'plant', 'property', 'purchases', 'sales', 'was', 'were', 'year']


### 3.3 Tính toán bảng tần suất từ - TF Matrix
In toàn bộ ma trận TF dưới dạng DataFrame để quan sát chi tiết từng từ.

In [13]:
tf_records = []
for doc_name, tokens in tokenized_docs.items():
    total_words = len(tokens)
    doc_tf = {}
    for word in vocabulary:
        count = tokens.count(word)
        doc_tf[word] = count / total_words  # Tần suất chuẩn hóa
    tf_records.append(doc_tf)

df_tf = pd.DataFrame(tf_records, index=corpus.keys())
print("[SUCCESS] Đã tính xong ma trận TF!")

# Hiển thị toàn bộ ma trận TF (không cắt bớt cột nào)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n=== TOÀN BỘ MA TRẬN TF (DOCUMENT-TERM MATRIX) ===")
print(df_tf.to_string())
df_tf

[SUCCESS] Đã tính xong ma trận TF!

=== TOÀN BỘ MA TRẬN TF (DOCUMENT-TERM MATRIX) ===
        $10959  $29760  $383285    $52729  $574785      2023  2024  acquisition    amazon       and     apple  equipment    fiscal       for        in  income   million  net  nvidia        of  payments     plant  property  purchases  sales  was      were      year
doc1  0.000000     0.0      0.1  0.000000      0.0  0.100000   0.0     0.000000  0.000000  0.000000  0.100000   0.000000  0.100000  0.000000  0.100000     0.0  0.100000  0.1     0.0  0.000000  0.000000  0.000000  0.000000   0.000000    0.1  0.0  0.100000  0.100000
doc2  0.083333     0.0      0.0  0.000000      0.0  0.000000   0.0     0.083333  0.000000  0.083333  0.083333   0.083333  0.000000  0.083333  0.000000     0.0  0.083333  0.0     0.0  0.083333  0.083333  0.083333  0.083333   0.000000    0.0  0.0  0.083333  0.000000
doc3  0.000000     0.0      0.0  0.000000      0.1  0.100000   0.0     0.000000  0.100000  0.000000  0.000000   0.00000

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.0,0.1,0.000000,0.0,0.100000,0.0,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.000000,0.100000,0.0,0.100000,0.1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.1,0.0,0.100000,0.100000
doc2,0.083333,0.0,0.0,0.000000,0.0,0.000000,0.0,0.083333,0.000000,0.083333,0.083333,0.083333,0.000000,0.083333,0.000000,0.0,0.083333,0.0,0.0,0.083333,0.083333,0.083333,0.083333,0.000000,0.0,0.0,0.083333,0.000000
doc3,0.000000,0.0,0.0,0.000000,0.1,0.100000,0.0,0.000000,0.100000,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.0,0.100000,0.1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.1,0.0,0.100000,0.100000
doc4,0.000000,0.0,0.0,0.076923,0.0,0.076923,0.0,0.000000,0.076923,0.076923,0.000000,0.076923,0.076923,0.000000,0.076923,0.0,0.076923,0.0,0.0,0.076923,0.000000,0.000000,0.076923,0.076923,0.0,0.0,0.076923,0.076923
doc5,0.000000,0.1,0.0,0.000000,0.0,0.000000,0.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.1,0.100000,0.1,0.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.1,0.000000,0.100000


### 3.4 Tính toán giá trị IDF của từng từ
In ra toàn bộ giá trị IDF của từ vựng trong corpus.

In [14]:
num_docs = len(corpus)
idf_dict = {}
for word in vocabulary:
    # Đếm số văn bản chứa từ word
    containing_docs = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    # Tính IDF với công thức smooth
    idf = np.log(num_docs / (1 + containing_docs)) # Smooth IDF
    idf_dict[word] = idf

df_idf = pd.Series(idf_dict).sort_values(ascending=False)
print("[SUCCESS] Đã tính xong IDF của toàn bộ từ vựng!")
print("\n=== TOÀN BỘ GIÁ TRỊ IDF CỦA TỪ VỰNG (Giảm dần) ===")
df_idf_df = pd.DataFrame({'Từ': df_idf.index, 'IDF': df_idf.values})
print(df_idf_df.to_string(index=False))
df_idf_df

[SUCCESS] Đã tính xong IDF của toàn bộ từ vựng!

=== TOÀN BỘ GIÁ TRỊ IDF CỦA TỪ VỰNG (Giảm dần) ===
         Từ       IDF
     $10959  0.916291
     $29760  0.916291
    $383285  0.916291
     $52729  0.916291
    $574785  0.916291
       2024  0.916291
acquisition  0.916291
        was  0.916291
   payments  0.916291
     nvidia  0.916291
     income  0.916291
        for  0.916291
      plant  0.916291
  purchases  0.916291
   property  0.510826
      apple  0.510826
     amazon  0.510826
  equipment  0.510826
        and  0.510826
         of  0.510826
      sales  0.510826
       2023  0.223144
        net  0.223144
     fiscal  0.000000
       were  0.000000
         in  0.000000
       year  0.000000
    million -0.182322


,Từ,IDF
0,$10959,0.916291
1,$29760,0.916291
2,$383285,0.916291
3,$52729,0.916291
4,$574785,0.916291
5,2024,0.916291
6,acquisition,0.916291
7,was,0.916291
8,payments,0.916291
9,nvidia,0.916291


### 3.5 Tính toán ma trận trọng số cuối cùng - TF-IDF Matrix
Nhân ma trận TF với vector IDF tương ứng.

In [15]:
tfidf_records = []
for doc_name, tokens in tokenized_docs.items():
    doc_tfidf = {}
    total_words = len(tokens)
    for word in vocabulary:
        tf = tokens.count(word) / total_words
        idf = idf_dict[word]
        doc_tfidf[word] = tf * idf
    tfidf_records.append(doc_tfidf)

df_tfidf = pd.DataFrame(tfidf_records, index=corpus.keys())
print("[SUCCESS] Đã dựng thành công ma trận TF-IDF!")

print("\n=== TOÀN BỘ MA TRẬN TF-IDF ===")
print(df_tfidf.to_string())
df_tfidf

[SUCCESS] Đã dựng thành công ma trận TF-IDF!

=== TOÀN BỘ MA TRẬN TF-IDF ===
        $10959    $29760   $383285    $52729   $574785      2023      2024  acquisition    amazon       and     apple  equipment  fiscal       for   in    income   million       net    nvidia        of  payments     plant  property  purchases     sales       was  were  year
doc1  0.000000  0.000000  0.091629  0.000000  0.000000  0.022314  0.000000     0.000000  0.000000  0.000000  0.051083   0.000000     0.0  0.000000  0.0  0.000000 -0.018232  0.022314  0.000000  0.000000  0.000000  0.000000  0.000000   0.000000  0.051083  0.000000   0.0   0.0
doc2  0.076358  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000     0.076358  0.000000  0.042569  0.042569   0.042569     0.0  0.076358  0.0  0.000000 -0.015193  0.000000  0.000000  0.042569  0.076358  0.076358  0.042569   0.000000  0.000000  0.000000   0.0   0.0
doc3  0.000000  0.000000  0.000000  0.000000  0.091629  0.022314  0.000000     0.000000  0.051083 

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.000000,0.091629,0.000000,0.000000,0.022314,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.000000,0.0,0.000000,-0.018232,0.022314,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.0
doc2,0.076358,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.076358,0.000000,0.042569,0.042569,0.042569,0.0,0.076358,0.0,0.000000,-0.015193,0.000000,0.000000,0.042569,0.076358,0.076358,0.042569,0.000000,0.000000,0.000000,0.0,0.0
doc3,0.000000,0.000000,0.000000,0.000000,0.091629,0.022314,0.000000,0.000000,0.051083,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,-0.018232,0.022314,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.0
doc4,0.000000,0.000000,0.000000,0.070484,0.000000,0.017165,0.000000,0.000000,0.039294,0.039294,0.000000,0.039294,0.0,0.000000,0.0,0.000000,-0.014025,0.000000,0.000000,0.039294,0.000000,0.000000,0.039294,0.070484,0.000000,0.000000,0.0,0.0
doc5,0.000000,0.091629,0.000000,0.000000,0.000000,0.000000,0.091629,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.091629,-0.018232,0.022314,0.091629,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.091629,0.0,0.0


## 4. Thực thi Tìm kiếm (Retrieval Process)
Nhận câu hỏi (Query) $\to$ Chuyển thành vector TF-IDF $\to$ Tính Cosine Similarity đối chiếu với toàn bộ văn bản trong ma trận TF-IDF.

In [16]:
def search_tfidf(query):
    print(f"\n=== CÂU HỎI USER: '{query}' ===")
    query_tokens = tokenize(query)
    print(f"[Step 1] Tokenized Query: {query_tokens}")
    
    # Dựng Vector TF-IDF cho Query
    query_vector = np.zeros(len(vocabulary))
    total_q_words = len(query_tokens) if len(query_tokens) > 0 else 1
    
    print("\n[Log] Chi tiết tính TF-IDF cho từng từ trong Query:")
    for word in query_tokens:
        if word in vocabulary:
            word_idx = vocabulary.index(word)
            tf = query_tokens.count(word) / total_q_words
            idf = idf_dict[word]
            query_vector[word_idx] = tf * idf
            print(f"  - Từ '{word}' (Index {word_idx}): tf = {query_tokens.count(word)}/{total_q_words} = {tf:.4f}, idf = {idf:.4f} -> tf-idf = {tf*idf:.4f}")
        else:
            print(f"  - Từ '{word}': Không nằm trong từ điển (Out of Vocabulary - OOV)")
            
    # In Vector Query đầy đủ
    print(f"\nVector Query TF-IDF (độ dài {len(vocabulary)} chiều):\n{query_vector}")
    
    # Tính Cosine Similarity với từng tài liệu
    q_norm = np.linalg.norm(query_vector)
    print(f"\n[Step 2] Độ dài Vector Query (L2 Norm): ||q|| = {q_norm:.4f}")
    
    scores = {}
    for doc_name in corpus.keys():
        doc_vector = df_tfidf.loc[doc_name].values
        doc_norm = np.linalg.norm(doc_vector)
        
        # Tính tích vô hướng (Dot Product) và in chi tiết các phép nhân từng chiều
        dot_product = 0.0
        multiplications = []
        for idx, word in enumerate(vocabulary):
            q_val = query_vector[idx]
            d_val = doc_vector[idx]
            if q_val > 0 and d_val > 0:
                prod = q_val * d_val
                dot_product += prod
                multiplications.append(f"('{word}': q_tfidf={q_val:.4f} * d_tfidf={d_val:.4f} = {prod:.4f})")
        
        # Tính cosine
        if q_norm > 0 and doc_norm > 0:
            cosine_sim = dot_product / (q_norm * doc_norm)
        else:
            cosine_sim = 0.0
            
        scores[doc_name] = cosine_sim
        print(f"\n* Đối chiếu với {doc_name}:")
        print(f"  - Vector của {doc_name}:\n    {doc_vector}")
        print(f"  - Độ dài vector ||d|| = {doc_norm:.4f}")
        print(f"  - Các phép nhân thành phần: {', '.join(multiplications) if multiplications else 'Không có từ khớp (tích = 0)'}")
        print(f"  - Tích vô hướng (Dot Product) = {dot_product:.4f}")
        print(f"  - Cosine Similarity = {dot_product:.4f} / ({q_norm:.4f} * {doc_norm:.4f}) = {cosine_sim:.4f}")
        
    # Sắp xếp xếp hạng
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\n=== KẾT QUẢ XẾP HẠNG RETRIEVAL ===")
    for rank, (doc_name, score) in enumerate(sorted_scores):
        print(f"  Hạng {rank+1}: {doc_name} | Score Cosine: {score:.4f} | Nội dung: {corpus[doc_name]}")
        
    return sorted_scores

## 5. Phân tích lỗi và Điểm yếu (Failure Analysis)

Bây giờ, chúng ta chạy hai câu hỏi kiểm thử để vạch rõ giới hạn của TF-IDF.

In [17]:
# Test 1: Khớp hoàn hảo từ khóa (Exact Keyword Match)
search_tfidf("Apple Net sales in 2023")


=== CÂU HỎI USER: 'Apple Net sales in 2023' ===
[Step 1] Tokenized Query: ['apple', 'net', 'sales', 'in', '2023']

[Log] Chi tiết tính TF-IDF cho từng từ trong Query:
  - Từ 'apple' (Index 10): tf = 1/5 = 0.2000, idf = 0.5108 -> tf-idf = 0.1022
  - Từ 'net' (Index 17): tf = 1/5 = 0.2000, idf = 0.2231 -> tf-idf = 0.0446
  - Từ 'sales' (Index 24): tf = 1/5 = 0.2000, idf = 0.5108 -> tf-idf = 0.1022
  - Từ 'in' (Index 14): tf = 1/5 = 0.2000, idf = 0.0000 -> tf-idf = 0.0000
  - Từ '2023' (Index 5): tf = 1/5 = 0.2000, idf = 0.2231 -> tf-idf = 0.0446

Vector Query TF-IDF (độ dài 28 chiều):
[0.         0.         0.         0.         0.         0.04462871
 0.         0.         0.         0.         0.10216512 0.
 0.         0.         0.         0.         0.         0.04462871
 0.         0.         0.         0.         0.         0.
 0.10216512 0.         0.         0.        ]

[Step 2] Độ dài Vector Query (L2 Norm): ||q|| = 0.1577

* Đối chiếu với doc1:
  - Vector của doc1:
    [ 0.   

[('doc1', np.float64(0.644898785818549)),
 ('doc3', np.float64(0.37411944104748696)),
 ('doc2', np.float64(0.14068266639609808)),
 ('doc4', np.float64(0.03606669664087338)),
 ('doc5', np.float64(0.030527167536609653))]

In [18]:
# Test 2: Gặp từ đồng nghĩa tài chính (Lexical Gap)
# Trong câu hỏi dùng từ 'capital expenditures' (chi tiêu vốn)
# Nhưng trong tài liệu của Amazon (doc4) dùng cụm 'purchases of property and equipment'
search_tfidf("Amazon capital expenditures 2023")


=== CÂU HỎI USER: 'Amazon capital expenditures 2023' ===
[Step 1] Tokenized Query: ['amazon', 'capital', 'expenditures', '2023']

[Log] Chi tiết tính TF-IDF cho từng từ trong Query:
  - Từ 'amazon' (Index 8): tf = 1/4 = 0.2500, idf = 0.5108 -> tf-idf = 0.1277
  - Từ 'capital': Không nằm trong từ điển (Out of Vocabulary - OOV)
  - Từ 'expenditures': Không nằm trong từ điển (Out of Vocabulary - OOV)
  - Từ '2023' (Index 5): tf = 1/4 = 0.2500, idf = 0.2231 -> tf-idf = 0.0558

Vector Query TF-IDF (độ dài 28 chiều):
[0.         0.         0.         0.         0.         0.05578589
 0.         0.         0.12770641 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ]

[Step 2] Độ dài Vector Query (L2 Norm): ||q|| = 0.1394

* Đối chiếu với doc1:
  - Vector của doc1:
    [ 0.          0.          0.09162907  0.          0.          0.02231436
  0.          0

[('doc3', np.float64(0.45601230463126696)),
 ('doc4', np.float64(0.31830544093258173)),
 ('doc1', np.float64(0.07307248284553076)),
 ('doc2', np.float64(0.0)),
 ('doc5', np.float64(0.0))]

### Nhận xét lỗi trên Test 2:
- Tài liệu đúng là **doc4** (`Amazon purchases of property and equipment...`).
- Kết quả xếp hạng: **doc4 nhận điểm số 0.0000** (đồng hạng bét với doc2, doc5).
- **Nguyên nhân:** Từ khóa chính `capital` và `expenditures` hoàn toàn không xuất hiện trong `doc4`. TF-IDF không có khả năng hiểu ngữ nghĩa đồng nghĩa, dẫn đến thất bại thu hồi thông tin hoàn toàn.